# 01 · Interpretability — Grad-CAM Activation Analysis

Apply Grad-CAM on `layer4[-1]` of ResNet-50 to visualise which facial regions the model focuses on when classifying a face-swap. Activations consistently concentrate in the mid-face region (eyes–nose), consistent with the model learning a distributed manipulation signature.

## Imports

In [ ]:
import torch
import torch.nn as nn
from torchvision import models
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

## Config

In [ ]:
IMG_SIZE   = 224
THRESHOLD  = 0.98
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
CHECKPOINT = './05 Checkpoints/best_model_r50.pth'
print(DEVICE)

## Pre-processing — eval transform

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

eval_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE, interpolation=cv2.INTER_LINEAR),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

## Model — ResNet-50 (`DeepfakeClassifier`)

In [ ]:
class DeepfakeClassifier(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        resnet = models.resnet50(
            weights=models.ResNet50_Weights.DEFAULT if pretrained else None
        )

        for p in resnet.parameters():
            p.requires_grad = False
        # for p in resnet.layer2.parameters():
        #     p.requires_grad = True
        for p in resnet.layer3.parameters():
            p.requires_grad = True
        for p in resnet.layer4.parameters():
            p.requires_grad = True

        self.conv1   = resnet.conv1
        self.bn1     = resnet.bn1
        self.relu    = resnet.relu
        self.maxpool = resnet.maxpool
        self.layer1  = resnet.layer1
        self.layer2  = resnet.layer2
        self.layer3  = resnet.layer3
        self.layer4  = resnet.layer4
        self.avgpool = resnet.avgpool

        self.head = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(2048, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2),
            nn.Linear(256, 1),
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        return self.head(x).squeeze(1)

## Load checkpoint

In [ ]:
def load_model(ckpt_path):
    model = DeepfakeClassifier(pretrained=False).to(DEVICE)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval()
    return model

model = load_model(CHECKPOINT)

## Grad-CAM — helper classes and visualisation function

In [ ]:
class BinaryClassifierOutputTarget:
    """Target para clasificadores binarios con un solo logit (shape [batch])."""
    def __call__(self, model_output):
        # model_output puede venir como [batch] o escalar tras squeeze
        if model_output.ndim == 0:
            return model_output
        return model_output[0]


def visualize_gradcam(model, image_path, device, threshold=0.98,
                      target_layer=None, save_path=None, title='Resnet 50'):
    """
    Genera Grad-CAM para una imagen y la visualiza con su score.
    """
    model.eval()

    if target_layer is None:
        target_layer = model.layer4[-1]

    img_rgb = np.array(Image.open(image_path).convert('RGB'))
    tensor  = eval_transform(image=img_rgb)['image'].unsqueeze(0).to(device)

    with torch.no_grad():
        logit = model(tensor)
        score = torch.sigmoid(logit).item()

    is_fraud   = score >= threshold
    label_text = 'FRAUDE (Deepfake)' if is_fraud else 'NO FRAUDE (Real)'
    color      = 'red' if is_fraud else 'green'

    # Grad-CAM con target explícito para clasificador binario
    cam = GradCAM(model=model, target_layers=[target_layer])
    targets = [BinaryClassifierOutputTarget()]
    grayscale_cam = cam(input_tensor=tensor, targets=targets)[0]

    img_resized = cv2.resize(img_rgb, (224, 224))
    img_float   = img_resized.astype(np.float32) / 255.0
    cam_overlay = show_cam_on_image(img_float, grayscale_cam, use_rgb=True)

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    axes[0].imshow(img_resized)
    axes[0].set_title('Original', fontsize=12)
    axes[0].axis('off')

    axes[1].imshow(cam_overlay)
    axes[1].set_title('Grad-CAM', fontsize=12)
    axes[1].axis('off')

    fig.suptitle(
        f'Score: {score:.4f}   |   Umbral: {threshold}   →   {label_text}\n{title}',
        fontsize=13, color=color, fontweight='bold',
    )

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()

    return score, is_fraud

## Run — single image

In [ ]:
score, is_fraud = visualize_gradcam(
    model,
    image_path='./03 All Data/52401_62209.png',
    device=DEVICE,
    threshold=THRESHOLD,
    save_path='./06 Figures/gradcam_r50.png',
)